<a href="https://colab.research.google.com/github/tiendrebeogoarlette97-bit/NLP-BERT-Sentiment-Classification/blob/main/notebooks/NLP_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification de sentiments : TF-IDF et BERT

**Objectif :** comparer une baseline TF-IDF + régression logistique à BERT-base-uncased fine-tuné sur SST-2 (GLUE). Les modèles seront évalués sur la même partition officielle de validation, avec accuracy, F1 et matrices de confusion. Les labels du test officiel ne sont pas publics. Aucune performance n'est présupposée.

## 1. Installation et configuration

Activer un GPU dans Colab : Exécution > Modifier le type d'exécution > T4 GPU. La seed fixe les générateurs aléatoires, sans garantir une reproductibilité bit à bit sur GPU.

In [ ]:
%pip install -q transformers==5.24.1 datasets==5.0.1 accelerate==1.16.0 pyarrow==23.0.1 jedi

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_dataset
from transformers import set_seed
from IPython.display import display

SEED = 42
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch :', torch.__version__)
print('Device :', DEVICE)
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
else:
    print('Attention : activer un GPU avant le fine-tuning.')

## 2. Chargement de SST-2

Nous conservons les partitions officielles. Le train sert à apprendre les paramètres et le vocabulaire TF-IDF ; la validation sert à comparer les modèles. Le test est inspecté uniquement pour sa structure.

In [ ]:
dataset = load_dataset('nyu-mll/glue', 'sst2')
print(dataset)
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()
label_names = {0: 'Negative', 1: 'Positive'}
display(train_df.head(10))
print('Labels du test :', test_df['label'].unique())

## 3. Analyse exploratoire des données

Nous vérifions la qualité des textes, les labels, les doublons et les longueurs. Les doublons sont recensés sans suppression automatique afin de préserver le protocole officiel ; les éventuels chevauchements train/validation seront signalés comme limite.

In [ ]:
summary = []
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    normalized = df['sentence'].fillna('').str.strip().str.lower()
    summary.append({'split': name, 'rows': len(df), 'missing_text': df['sentence'].isna().sum(), 'empty_text': normalized.eq('').sum(), 'duplicate_texts': normalized.duplicated().sum(), 'label_values': sorted(df['label'].unique().tolist())})
display(pd.DataFrame(summary))
assert train_df['label'].isin([0, 1]).all()
assert val_df['label'].isin([0, 1]).all()
train_texts = set(train_df['sentence'].str.strip().str.lower())
val_texts = set(val_df['sentence'].str.strip().str.lower())
print('Textes distincts communs train/validation :', len(train_texts & val_texts))

In [ ]:
eda = pd.concat([train_df.assign(split='train'), val_df.assign(split='validation')], ignore_index=True)
eda['sentiment'] = eda['label'].map(label_names)
eda['word_count'] = eda['sentence'].str.split().str.len()
counts = pd.crosstab(eda['split'], eda['sentiment'])
display(counts)
display(pd.crosstab(eda['split'], eda['sentiment'], normalize='index').mul(100).round(2))
display(eda.groupby('split')['word_count'].describe(percentiles=[.5, .9, .95, .99]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, split in zip(axes, ['train', 'validation']):
    sns.countplot(data=eda[eda['split'] == split], x='sentiment', order=['Negative', 'Positive'], ax=ax)
    ax.set_title(f'Distribution des classes — {split}')
    ax.set_xlabel('Sentiment')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(data=eda, x='word_count', hue='split', stat='density', common_norm=False, element='step', bins=40)
plt.xlabel('Nombre de mots par phrase')
plt.title('Distribution des longueurs de phrases')
plt.tight_layout()
plt.show()

### Interprétation de l'EDA

Résultats observés dans Colab : 67 349 exemples train, 872 validation et 1 821 test. Aucun texte manquant ou vide. Le train contient 371 occurrences redondantes après normalisation (environ 0,55 %), contre aucune en validation et test. Aucun chevauchement textuel exact normalisé train/validation n'a été détecté ; cela n'exclut pas les quasi-doublons. Les classes positives représentent 55,78 % du train et 50,92 % de la validation. Nous conservons les partitions officielles sans rééquilibrage. Les textes de validation sont plus longs (19,55 mots en moyenne contre 9,41), ce qui motive une analyse ultérieure des performances par longueur. Les labels test (-1) ne sont pas publics. Les résultats rapportés concernent la validation officielle.

In [ ]:
duplicate_check = train_df.assign(normalized=train_df['sentence'].str.strip().str.lower()).groupby('normalized')['label'].agg(['size', 'nunique'])
print('Groupes de textes répétés :', int((duplicate_check['size'] > 1).sum()))
print('Groupes avec labels contradictoires :', int((duplicate_check['nunique'] > 1).sum()))

## 4. Baseline : TF-IDF + régression logistique

Le vectoriseur apprend son vocabulaire et ses poids IDF exclusivement sur le train. Nous conservons les négations, utilisons des unigrams et bigrams, et n'appliquons ni suppression de stopwords ni stemming. La régression logistique utilise une régularisation L2 (C=1) et une configuration fixe, sans optimisation sur la validation officielle. Le pipeline empêche la réestimation du vocabulaire sur la validation.

In [ ]:
from time import perf_counter
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay, log_loss
from sklearn.dummy import DummyClassifier

X_train = train_df['sentence'].tolist()
y_train = train_df['label'].to_numpy()
X_val = val_df['sentence'].tolist()
y_val = val_df['label'].to_numpy()

baseline = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True, norm='l2', token_pattern=r'(?u)\b\w+\b')),
    ('lr', LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=SEED))
])
t0 = perf_counter()
baseline.fit(X_train, y_train)
baseline_train_time = perf_counter() - t0
t0 = perf_counter()
baseline_proba = baseline.predict_proba(X_val)
baseline_pred = baseline.classes_[baseline_proba.argmax(axis=1)]
baseline_inference_time = perf_counter() - t0
print('Taille du vocabulaire :', len(baseline.named_steps['tfidf'].vocabulary_))
print('Iterations LR :', baseline.named_steps['lr'].n_iter_)
print('Temps entrainement (s) :', round(baseline_train_time, 3))

In [ ]:
def classification_metrics(y_true, y_pred):
    return {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0), 'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0)}

baseline_metrics = classification_metrics(y_val, baseline_pred)
baseline_metrics.update({'train_time_s': baseline_train_time, 'inference_time_s': baseline_inference_time, 'validation_log_loss': log_loss(y_val, baseline_proba, labels=[0,1])})
results = pd.DataFrame([{'model': 'TF-IDF + LR', **baseline_metrics}])
display(results.round(4))
print(classification_report(y_val, baseline_pred, target_names=['Negative', 'Positive'], digits=4))
print('Accuracy train :', accuracy_score(y_train, baseline.predict(X_train)))
dummy = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(y_train), 1)), y_train)
print('Accuracy classe majoritaire :', dummy.score(np.zeros((len(y_val), 1)), y_val))
ConfusionMatrixDisplay.from_predictions(y_val, baseline_pred, display_labels=['Negative', 'Positive'], cmap='Blues')
plt.title('Matrice de confusion — TF-IDF + LR')
plt.tight_layout()
plt.show()

### Interprétabilité de la baseline

Les coefficients positifs et négatifs sont des associations apprises avec la classe positive et négative respectivement. Ils ne prouvent pas une relation causale.

In [ ]:
features = np.array(baseline.named_steps['tfidf'].get_feature_names_out())
weights = baseline.named_steps['lr'].coef_[0]
order = np.argsort(weights)
selected = np.r_[order[:15], order[-15:]]
coef_df = pd.DataFrame({'term': features[selected], 'coefficient': weights[selected]}).sort_values('coefficient')
plt.figure(figsize=(9, 10))
plt.barh(coef_df['term'], coef_df['coefficient'], color=np.where(coef_df['coefficient'] > 0, 'seagreen', 'indianred'))
plt.axvline(0, color='black', linewidth=.8)
plt.title('Termes les plus associés aux sentiments — régression logistique')
plt.xlabel('Coefficient appris')
plt.tight_layout()
plt.show()
val_predictions = val_df[['idx', 'sentence', 'label']].copy()
val_predictions['baseline_pred'] = baseline_pred
val_predictions['baseline_prob_positive'] = baseline_proba[:, 1]
display(val_predictions[val_predictions['label'] != val_predictions['baseline_pred']].head(10))

### Résultats observés — baseline initiale

Sur SST-2 validation : accuracy 0,8096 ; F1 positif 0,8223 ; F1 macro 0,8087 ; précision positive 0,7837 ; rappel positif 0,8649. Matrice (labels 0,1) : [[322,106],[60,384]]. Accuracy train : 0,9540. Le gap de 14,44 points suggère un problème de généralisation, sans prouver à lui seul un surapprentissage (les distributions de longueurs diffèrent). Les faux positifs sont plus nombreux que les faux négatifs. Ces observations motivent l'expérience complémentaire suivante.

## 5. Optimisation de la baseline (expérience complémentaire)

La validation officielle n'intervient pas dans la recherche d'hyperparamètres. Nous utilisons StratifiedGroupKFold avec trois folds et des groupes de textes identiques après normalisation, pour limiter la fuite entre folds due aux doublons. Le pipeline ajuste TF-IDF à l'intérieur de chaque fold. Le F1-macro moyen sélectionne la configuration ; l'accuracy et les scores train servent au diagnostic. La grille comprend 40 configurations et 120 ajustements internes. Nous conservons la baseline initiale et rapportons aussi le modèle optimisé, même si ce dernier obtient un score inférieur en validation officielle. Comme cette expérience a été motivée par l'observation de la validation officielle, celle-ci ne constitue pas un test indépendant entièrement aveugle.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.base import clone
from joblib import Memory
from tempfile import TemporaryDirectory

# Group identical normalized texts within the same fold.
groups = train_df['sentence'].str.strip().str.lower().to_numpy()
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'lr__C': [0.1, 0.5, 1.0, 5.0, 10.0],
    'lr__class_weight': [None, 'balanced']
}

# Cache vectorization within each fold to avoid repeating identical TF-IDF work.
# Limit parallel jobs to control Colab RAM use.
with TemporaryDirectory(prefix='sst2_tfidf_') as cache_dir:
    search_pipeline = clone(baseline).set_params(memory=Memory(cache_dir, verbose=0))
    search = GridSearchCV(
        search_pipeline, param_grid, cv=cv,
        scoring={'f1_macro': 'f1_macro', 'accuracy': 'accuracy'},
        refit='f1_macro', return_train_score=True,
        n_jobs=2, pre_dispatch=2, verbose=1, error_score='raise'
    )
    t0 = perf_counter()
    search.fit(X_train, y_train, groups=groups)
    tuning_time = perf_counter() - t0
    optimized_baseline = search.best_estimator_.set_params(memory=None)

print('Meilleurs hyperparamètres :', search.best_params_)
print('F1-macro CV :', round(search.best_score_, 4))
print('Temps recherche + refit (s) :', round(tuning_time, 2))
cv_results = pd.DataFrame(search.cv_results_)
cv_results['cv_gap_f1_macro'] = cv_results['mean_train_f1_macro'] - cv_results['mean_test_f1_macro']
cols = ['param_tfidf__ngram_range', 'param_tfidf__min_df', 'param_lr__C', 'param_lr__class_weight', 'mean_train_f1_macro', 'mean_test_f1_macro', 'std_test_f1_macro', 'mean_test_accuracy', 'cv_gap_f1_macro', 'rank_test_f1_macro']
display(cv_results.sort_values('rank_test_f1_macro')[cols].head(10).round(4))
initial_cv = cv_results[(cv_results['param_tfidf__ngram_range'].map(tuple) == (1, 2)) & (cv_results['param_tfidf__min_df'] == 2) & (cv_results['param_lr__C'] == 1.0) & cv_results['param_lr__class_weight'].isna()]
print('Configuration initiale — CV :')
display(initial_cv[cols].round(4))

### Évaluation de la baseline optimisée

Le meilleur modèle est choisi uniquement via la CV interne. Nous le réentraînons sans cache pour mesurer un temps de fit comparable à celui de la baseline initiale. Les métriques officielles et les écarts sont ensuite calculés ; aucune amélioration n'est garantie.

In [ ]:
optimized_baseline = clone(baseline).set_params(**search.best_params_)
t0 = perf_counter()
optimized_baseline.fit(X_train, y_train)
optimized_train_time = perf_counter() - t0
t0 = perf_counter()
optimized_proba = optimized_baseline.predict_proba(X_val)
optimized_pred = optimized_baseline.classes_[optimized_proba.argmax(axis=1)]
optimized_inference_time = perf_counter() - t0
optimized_metrics = classification_metrics(y_val, optimized_pred)
optimized_metrics.update({'train_time_s': optimized_train_time, 'inference_time_s': optimized_inference_time, 'validation_log_loss': log_loss(y_val, optimized_proba, labels=[0,1])})
results = pd.DataFrame([{'model': 'TF-IDF + LR', **baseline_metrics}, {'model': 'TF-IDF + LR optimized', **optimized_metrics}])
display(results.round(4))
print(classification_report(y_val, optimized_pred, target_names=['Negative', 'Positive'], digits=4))
print('Vocabulaire initial / optimisé :', len(baseline.named_steps['tfidf'].vocabulary_), len(optimized_baseline.named_steps['tfidf'].vocabulary_))

from sklearn.metrics import confusion_matrix
comparison = []
for name, model, pred in [('Initial', baseline, baseline_pred), ('Optimized', optimized_baseline, optimized_pred)]:
    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, pred)
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0,1]).ravel()
    comparison.append({'model':name, 'train_accuracy':train_acc, 'val_accuracy':val_acc, 'gap':train_acc-val_acc, 'false_positives':fp, 'false_negatives':fn, 'FPR':fp/(fp+tn), 'FNR':fn/(fn+tp)})
display(pd.DataFrame(comparison).round(4))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name, pred in zip(axes, ['Initial', 'Optimized'], [baseline_pred, optimized_pred]):
    ConfusionMatrixDisplay.from_predictions(y_val, pred, labels=[0,1], display_labels=['Negative','Positive'], cmap='Blues', colorbar=False, ax=ax)
    ax.set_title(name)
plt.tight_layout()
plt.show()
val_predictions['optimized_pred'] = optimized_pred
val_predictions['optimized_prob_positive'] = optimized_proba[:,1]

### Analyse de la généralisation par longueur

Les groupes de longueur sont fixés d'après des seuils simples (10 et 20 mots). Nous rapportons les effectifs et le F1-macro par groupe, sans utiliser ces résultats pour ajuster les modèles.

In [ ]:
val_predictions['word_count'] = val_predictions['sentence'].str.split().str.len()
val_predictions['length_group'] = pd.cut(val_predictions['word_count'], bins=[0,10,20,np.inf], labels=['1-10','11-20','21+'])
length_results = []
for group, frame in val_predictions.groupby('length_group', observed=True):
    for name, col in [('Initial','baseline_pred'), ('Optimized','optimized_pred')]:
        length_results.append({'length_group':str(group), 'model':name, 'n':len(frame), 'n_negative':int((frame['label']==0).sum()), 'n_positive':int((frame['label']==1).sum()), 'accuracy':accuracy_score(frame['label'],frame[col]), 'f1_macro':f1_score(frame['label'],frame[col],labels=[0,1],average='macro',zero_division=0)})
length_results = pd.DataFrame(length_results)
display(length_results.round(4))
plt.figure(figsize=(8,4))
sns.barplot(data=length_results, x='length_group', y='accuracy', hue='model')
plt.ylim(0,1)
plt.xlabel('Nombre de mots')
plt.title('Accuracy de validation selon la longueur')
plt.tight_layout()
plt.show()

### Interprétation à compléter

Comparer F1-macro CV, écart train/validation, FPR/FNR et scores par longueur. Une diminution du gap n'est pas suffisante si elle résulte d'une baisse simultanée des performances. Une pondération équilibrée ne garantit pas des taux d'erreur identiques. Le choix final issu de la CV reste fixé indépendamment du score officiel.

### Résultats observés de l'optimisation

La CV sélectionne C=10, ngram_range=(1,2), min_df=2 et class_weight=None. F1-macro CV : 0,9239 contre 0,9032 initialement. Sur validation officielle, accuracy 0,8211 contre 0,8096 ; F1-macro 0,8203 contre 0,8087 ; log loss 0,3860 contre 0,3989. La matrice optimisée est [[329,99],[57,387]], soit 156 erreurs contre 166. Le gap d'accuracy train/validation augmente de 0,1444 à 0,1629 : le risque de surapprentissage n'est pas résolu. Le modèle optimisé utilise une régularisation plus faible et le même vocabulaire (74 521 termes). La CV interne ne reproduit pas nécessairement la distribution officielle et des fragments apparentés peuvent traverser les folds malgré le regroupement des doublons exacts. Nous conservons les deux baselines et utilisons celle optimisée par CV comme référence principale, sans nouvelle sélection sur la validation officielle. Les résultats par longueur et les doublons contradictoires restent à documenter après inspection des sorties.

## 5 bis. Diagnostic structurel de SST-2 (avant BERT)

Références : Socher et al. (2013), https://aclanthology.org/D13-1170/ ; dépôt Stanford https://github.com/stanfordnlp/sentiment-treebank ; Roberts et al. (2017), https://doi.org/10.1111/ecog.02881. Nous utilisons uniquement les arbres originaux d'entraînement. Chaque fragment GLUE est associé à tous les arbres contenant exactement sa séquence de feuilles normalisée. Les correspondances multiples sont conservées, pas résolues arbitrairement. Les arbres partageant un fragment sont reliés en composantes connexes pour diagnostiquer la faisabilité d'un découpage strict. Des fragments très fréquents peuvent créer une composante géante ; dans ce cas, une CV par composantes n'est pas utilisable et nous testerons une alternative par arbres avec purge des fragments partagés. Les résultats de couverture doivent être inspectés avant toute sélection de modèle.

In [ ]:
import re
import unicodedata
import requests
from collections import defaultdict, Counter
from nltk.tree import Tree

TREE_URL = 'https://raw.githubusercontent.com/stanfordnlp/sentiment-treebank/master/fiveclass/train.txt'
response = requests.get(TREE_URL, timeout=90)
response.raise_for_status()
lines = [line for line in response.text.splitlines() if line.strip()]
trees = [Tree.fromstring(line) for line in lines]
print('Arbres train Stanford :', len(trees))
assert len(trees) > 1000, 'Source inattendue : vérifier le fichier original.'

def normalize_sst(text):
    text = unicodedata.normalize('NFKC', text).lower()
    for a,b in {'-lrb-':'(', '-rrb-':')', '-lsb-':'[', '-rsb-':']', '-lcb-':'{', '-rcb-':'}'}.items():
        text = text.replace(a,b)
    return ' '.join(text.split())

phrase_to_trees = defaultdict(set)
root_to_trees = defaultdict(set)
for tree_id, tree in enumerate(trees):
    root_to_trees[normalize_sst(' '.join(tree.leaves()))].add(tree_id)
    for subtree in tree.subtrees():
        phrase_to_trees[normalize_sst(' '.join(subtree.leaves()))].add(tree_id)

train_keys = train_df['sentence'].map(normalize_sst)
parent_sets = [phrase_to_trees.get(key, set()) for key in train_keys]
root_sets = [root_to_trees.get(key, set()) for key in train_keys]
match_counts = np.array([len(s) for s in parent_sets])
root_counts = np.array([len(s) for s in root_sets])
print('Correspondances exactes, normalisées, par nombre de parents :')
display(pd.Series(np.select([match_counts==0,match_counts==1],['aucun','unique'],default='multiple')).value_counts().to_frame('n'))
print('Couverture :', (match_counts>0).mean())
print('Observations correspondant à une racine :', (root_counts>0).sum())
display(train_df.loc[match_counts==0,['idx','sentence','label']].head(15))
print('SHA256 source :', __import__('hashlib').sha256(response.content).hexdigest())

In [ ]:
class UnionFind:
    def __init__(self,n): self.parent=list(range(n)); self.size=[1]*n
    def find(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]]; x=self.parent[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b: return
        if self.size[a]<self.size[b]: a,b=b,a
        self.parent[b]=a; self.size[a]+=self.size[b]

uf=UnionFind(len(trees))
for parents in parent_sets:
    if len(parents)>1:
        first=min(parents)
        for other in parents: uf.union(first,other)
component_counts=Counter(uf.find(i) for i in range(len(trees)))
matched_row_components=Counter(uf.find(min(s)) for s in parent_sets if s)
print('Nombre de composantes arbres :',len(component_counts))
print('Plus grandes composantes (arbres) :',component_counts.most_common(10))
print('Plus grandes composantes (lignes GLUE associées) :',matched_row_components.most_common(10))
print('Part maximale des lignes train dans une composante :',max(matched_row_components.values(),default=0)/len(train_df))
print('Aucun identifiant de parent n’est inventé pour les textes non associés.')

### Alternative : validation par arbres avec purge

Nous divisons les identifiants d'arbres, puis retenons seulement les observations dont tous les parents connus se trouvent exclusivement du même côté. Les lignes ambiguës traversant la frontière et les lignes non associées sont exclues de ce diagnostic interne. La validation interne principale porte sur les racines exclusivement associées au côté validation ; nous mesurons également les fragments retenus. Ce protocole est conservateur, peut modifier la distribution et ne constitue pas une preuve d'absence de quasi-doublons. La partition officielle GLUE reste inchangée.

In [ ]:
from sklearn.model_selection import KFold
structural_folds=[]
fold_summary=[]
for fold,(tr_ids,va_ids) in enumerate(KFold(n_splits=3,shuffle=True,random_state=SEED).split(trees),1):
    tr_set,va_set=set(tr_ids),set(va_ids)
    tr_idx=np.array([j for j,s in enumerate(parent_sets) if s and s.issubset(tr_set)],dtype=int)
    va_all=np.array([j for j,s in enumerate(parent_sets) if s and s.issubset(va_set)],dtype=int)
    va_idx=np.array([j for j in va_all if root_sets[j] and root_sets[j].issubset(va_set)],dtype=int)
    assert not set(train_keys.iloc[tr_idx]) & set(train_keys.iloc[va_idx])
    structural_folds.append((tr_idx,va_idx))
    fold_summary.append({'fold':fold,'train_rows':len(tr_idx),'validation_fragments_and_roots':len(va_all),'validation_roots':len(va_idx),'excluded_or_crossing':len(train_df)-len(tr_idx)-len(va_all),'train_positive_rate':y_train[tr_idx].mean() if len(tr_idx) else np.nan,'val_positive_rate':y_train[va_idx].mean() if len(va_idx) else np.nan,'val_mean_words':train_df.iloc[va_idx]['sentence'].str.split().str.len().mean()})
display(pd.DataFrame(fold_summary).round(4))
assert all(len(t)>0 and len(v)>0 and len(np.unique(y_train[t]))==2 and len(np.unique(y_train[v]))==2 for t,v in structural_folds), 'Folds insuffisants : examiner la couverture.'

### Comparaison diagnostique des deux modèles existants

Les deux configurations déjà fixées (C=1 et C=10) sont évaluées sur les mêmes folds purgés. Nous ne modifions pas encore les hyperparamètres ni le train officiel. Comparer les scores avec la CV précédente, en tenant compte du changement simultané de composition des folds et du nombre de données.

In [ ]:
from sklearn.base import clone
structural_results=[]
for fold,(tr,va) in enumerate(structural_folds,1):
    for name,C in [('Initial',1.0),('Optimized',10.0)]:
        estimator=clone(baseline).set_params(lr__C=C)
        estimator.fit([X_train[j] for j in tr],y_train[tr])
        train_pred=estimator.predict([X_train[j] for j in tr])
        val_pred=estimator.predict([X_train[j] for j in va])
        train_f1=f1_score(y_train[tr],train_pred,average='macro')
        val_f1=f1_score(y_train[va],val_pred,average='macro')
        structural_results.append({'fold':fold,'model':name,'train_f1_macro':train_f1,'val_f1_macro':val_f1,'gap':train_f1-val_f1,'val_accuracy':accuracy_score(y_train[va],val_pred)})
structural_results=pd.DataFrame(structural_results)
display(structural_results.round(4))
display(structural_results.groupby('model')[['train_f1_macro','val_f1_macro','gap','val_accuracy']].agg(['mean','std']).round(4))

### Diagnostic structurel — résultats observés

8 527 arbres, 58 610 correspondances uniques, 4 031 multiples, 4 708 absentes (couverture 93,01 %). La composante géante contient 61 267 lignes (90,97 % du train), ce qui motive la purge. F1 macro structurel moyen : 0,7987 (C=1) et 0,8104 (C=10), contre 0,9032 et 0,9239 en CV précédente. Les gaps structurels moyens sont 0,1545 et 0,1731 environ. Les validations structurelles portent sur des racines et utilisent moins de données : la baisse ne permet pas d'isoler une fuite entre fragments. Les scores officiels sont 0,8087 et 0,8203. Par longueur (1–10, 11–20, 21+ mots), les accuracies initiales sont 0,8421, 0,8441, 0,7658 ; optimisées 0,8882, 0,8382, 0,7789. Le risque de surapprentissage persiste.

## 5 ter. Étude de la complexité et de la régularisation

Protocole fixé avant exécution : comparer les candidats uniquement sur les trois folds structurels existants. Les références C=1 et C=10 sont incluses. Étudier séparément la limitation fréquentielle, la sélection chi-deux sur comptages, L1 et Elastic Net. La règle de sélection retient les candidats à moins d'une erreur standard du meilleur F1 macro moyen, puis le plus petit gap interne ; en cas d'égalité, le moins de coefficients non nuls. Ce choix est un compromis exploratoire, pas un estimateur indépendant du score final. Le score officiel ne sert pas à sélectionner. Les modèles non convergés sont exclus de la sélection. La partition officielle a déjà été consultée auparavant : elle n'est pas un test aveugle.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.exceptions import ConvergenceWarning
import warnings

candidates = []
for C in [0.1, 0.5, 1., 5., 10.]:
    candidates.append((f'L2 C={C:g}', clone(baseline).set_params(lr__C=C)))
for k in [5000,10000,25000,50000]:
    candidates.append((f'Max features {k}', clone(baseline).set_params(tfidf__max_features=k,lr__C=10.)))
for k in [10000,25000]:
    candidates.append((f'Chi2 {k}',Pipeline([('count',CountVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,token_pattern=r'(?u)\b\w+\b')),('select',SelectKBest(chi2,k=k)),('tfidf',TfidfTransformer(sublinear_tf=True,norm='l2')),('lr',LogisticRegression(C=10.,solver='lbfgs',max_iter=1000,random_state=SEED))])))
for C in [1.,10.]:
    candidates.append((f'L1 C={C:g}',clone(baseline).set_params(lr__solver='liblinear',lr__penalty='l1',lr__C=C,lr__max_iter=2000)))
    candidates.append((f'ElasticNet C={C:g}',clone(baseline).set_params(lr__solver='saga',lr__penalty='elasticnet',lr__l1_ratio=0.5,lr__C=C,lr__max_iter=1500,lr__tol=1e-3)))

complexity_rows=[]
for name, prototype in candidates:
    for fold,(tr,va) in enumerate(structural_folds,1):
        estimator=clone(prototype)
        xtr=[X_train[j] for j in tr]; xva=[X_train[j] for j in va]
        t0=perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always',ConvergenceWarning)
            estimator.fit(xtr,y_train[tr])
        elapsed=perf_counter()-t0
        converged=not any(issubclass(w.category,ConvergenceWarning) for w in caught)
        a=f1_score(y_train[tr],estimator.predict(xtr),average='macro')
        pred=estimator.predict(xva)
        b=f1_score(y_train[va],pred,average='macro')
        tn,fp,fn,tp=confusion_matrix(y_train[va],pred,labels=[0,1]).ravel()
        coef=estimator.named_steps['lr'].coef_
        complexity_rows.append(dict(model=name,fold=fold,train_f1=a,val_f1=b,gap=a-b,accuracy=accuracy_score(y_train[va],pred),features=coef.size,nonzero=np.count_nonzero(coef),fpr=fp/(fp+tn),fnr=fn/(fn+tp),seconds=elapsed,converged=converged))
    print('Terminé :',name,flush=True)
complexity_results=pd.DataFrame(complexity_rows)
summary_complexity=complexity_results.groupby('model').agg(val_f1=('val_f1','mean'),std=('val_f1','std'),train_f1=('train_f1','mean'),gap=('gap','mean'),accuracy=('accuracy','mean'),features=('features','mean'),nonzero=('nonzero','mean'),fpr=('fpr','mean'),fnr=('fnr','mean'),seconds=('seconds','mean'),converged=('converged','all')).sort_values('val_f1',ascending=False)
display(summary_complexity.round(4))

In [ ]:
eligible=summary_complexity[summary_complexity['converged']]
assert not eligible.empty
best=eligible.iloc[0]
threshold=best['val_f1']-best['std']/np.sqrt(len(structural_folds))
shortlist=eligible[eligible['val_f1']>=threshold].sort_values(['gap','nonzero','val_f1'],ascending=[True,True,False])
selected_name=shortlist.index[0]
selected_prototype=dict(candidates)[selected_name]
print('Meilleur F1 structurel :',eligible.index[0],best['val_f1'])
print('Seuil une erreur standard :',threshold)
display(shortlist.round(4))
print('Compromis retenu :',selected_name)
reg=summary_complexity.loc[[f'L2 C={C:g}' for C in [0.1,0.5,1.,5.,10.]]]
plt.figure(figsize=(8,4))
plt.plot([0.1,0.5,1,5,10],reg['train_f1'],'o-',label='Train')
plt.errorbar([0.1,0.5,1,5,10],reg['val_f1'],yerr=reg['std'],fmt='o-',label='Validation structurelle')
plt.xscale('log');plt.xlabel('C');plt.ylabel('F1 macro');plt.legend();plt.grid(alpha=.2);plt.show()
plt.figure(figsize=(8,5))
plt.scatter(eligible['gap'],eligible['val_f1'])
for name,row in eligible.iterrows():
    plt.annotate(name,(row['gap'],row['val_f1']),fontsize=7)
plt.xlabel('Gap F1 macro');plt.ylabel('F1 macro structurel');plt.tight_layout();plt.show()

### Courbes d'apprentissage structurelles

Pour chaque fold, le sous-échantillonnage des lignes d'entraînement est imbriqué et reproductible ; aucune ligne de validation n'entre dans le fit. Le nombre de lignes est réduit, pas le nombre de familles d'arbres de façon contrôlée. Comparer le modèle initial et le compromis retenu. Les courbes sont descriptives et ne constituent pas une nouvelle règle de sélection.

In [ ]:
learning_rows=[]
for fold,(tr,va) in enumerate(structural_folds,1):
    perm=np.random.default_rng(SEED+fold).permutation(tr)
    for fraction in [.1,.25,.5,.75,1.]:
        sub=perm[:max(2,int(len(perm)*fraction))]
        for name,proto in [('Initial',baseline),('Selected',selected_prototype)]:
            estimator=clone(proto)
            # Chi2 k cannot exceed the available vocabulary in small subsets.
            if 'select' in estimator.named_steps:
                counts=clone(estimator.named_steps['count']).fit([X_train[j] for j in sub])
                estimator.set_params(select__k=min(estimator.named_steps['select'].k,len(counts.vocabulary_)))
            xs=[X_train[j] for j in sub]; xv=[X_train[j] for j in va]
            estimator.fit(xs,y_train[sub])
            learning_rows.append({'fold':fold,'fraction':fraction,'model':name,'train_f1':f1_score(y_train[sub],estimator.predict(xs),average='macro'),'val_f1':f1_score(y_train[va],estimator.predict(xv),average='macro')})
learning_results=pd.DataFrame(learning_rows)
learning_summary=learning_results.groupby(['model','fraction'])[['train_f1','val_f1']].mean().reset_index()
display(learning_summary.round(4))
fig,axes=plt.subplots(1,2,figsize=(11,4),sharey=True)
for ax,(name,frame) in zip(axes,learning_summary.groupby('model')):
    ax.plot(frame['fraction'],frame['train_f1'],'o-',label='Train')
    ax.plot(frame['fraction'],frame['val_f1'],'o-',label='Validation structurelle')
    ax.set_title(name);ax.set_xlabel('Fraction des données');ax.set_ylabel('F1 macro');ax.legend()
plt.tight_layout();plt.show()

Arrêt de cette étape : inspecter les résultats, les avertissements de convergence et les courbes avant de réentraîner le compromis sur le train officiel. Aucun nouveau score officiel n'est calculé dans cette section.

## 6. Tokenisation BERT

Nous chargeons le tokenizer du checkpoint bert-base-uncased, sans appliquer le prétraitement TF-IDF. Nous inspectons WordPiece, les identifiants, les tokens spéciaux et les masques. Les longueurs incluent les tokens spéciaux ; la limite de 128 tokens est un choix initial, dont nous mesurons le taux de troncature avant entraînement.

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
MODEL_NAME = 'bert-base-uncased'
MAX_LENGTH = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
example = 'This movie was not good, but the cinematography was unforgettable.'
encoded = tokenizer(example, padding='max_length', max_length=32, truncation=True, return_attention_mask=True)
display(pd.DataFrame({'token': tokenizer.convert_ids_to_tokens(encoded['input_ids']), 'input_id': encoded['input_ids'], 'attention_mask': encoded['attention_mask']}))
print('Taille du vocabulaire du tokenizer :', len(tokenizer))

In [ ]:
token_length_rows = []
for split in ['train', 'validation']:
    lengths = []
    for batch in dataset[split].iter(batch_size=512):
        ids = tokenizer(batch['sentence'], add_special_tokens=True, truncation=False, verbose=False)['input_ids']
        lengths.extend(map(len, ids))
    lengths = np.asarray(lengths)
    token_length_rows.append({'split':split,'n':len(lengths),'mean':lengths.mean(),'median':np.median(lengths),'p95':np.percentile(lengths,95),'p99':np.percentile(lengths,99),'max':lengths.max(),'truncated_at_128':int((lengths>MAX_LENGTH).sum()),'truncation_rate':float((lengths>MAX_LENGTH).mean())})
display(pd.DataFrame(token_length_rows).round(4))

In [ ]:
def tokenize_batch(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=MAX_LENGTH)

tokenized = dataset.map(tokenize_batch, batched=True, remove_columns=['sentence','idx'])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None)
example_batch = data_collator([tokenized['train'][i] for i in range(4)])
print(tokenized)
print({key:tuple(value.shape) for key,value in example_batch.items()})
print('Labels du batch :', example_batch['labels'].tolist())

Le padding dynamique complète uniquement jusqu'à la longueur maximale du lot (arrondie au multiple de 8 sur GPU), ce qui réduit le calcul inutile. Le jeu test non étiqueté ne sera pas utilisé pour calculer les performances. Le fine-tuning sera ajouté après vérification de ces sorties.